# Prompt Injection Detection Final Project

In [1]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["ANTHROPIC_API_KEY"] = secrets.get_secret("project-key")

In [2]:
!pip install anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 481.8/481.8 kB 23.3 MB/s eta 0:00:00


## 1. Imports

In [3]:
import re
import random
import unicodedata

import pandas as pd
from datasets import load_dataset

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix,
)

random.seed(42)

## 2. Data Loading

In [4]:
DATA_FILES = {
    "train": "https://huggingface.co/datasets/jayavibhav/prompt-injection/resolve/main/data/train-00000-of-00001.parquet",
    "test":  "https://huggingface.co/datasets/jayavibhav/prompt-injection/resolve/main/data/test-00000-of-00001.parquet",
}

raw = load_dataset("parquet", data_files=DATA_FILES)

train_df = raw["train"].to_pandas().rename(columns={"text": "raw_text"})
test_df  = raw["test"].to_pandas().rename(columns={"text": "raw_text"})

print(f"Train: {len(train_df):,}  |  Test: {len(test_df):,}")
print(train_df["label"].value_counts(normalize=True).rename("proportion"))

data/train-00000-of-00001.parquet:   0%|          | 0.00/61.8M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/15.5M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Train: 261,738  |  Test: 65,416
label
0    0.506254
1    0.493746
Name: proportion, dtype: float64


## 3. Preprocessing

Applied to every split:
- NFKC Unicode normalisation (neutralises homoglyph attacks)
- Whitespace collapse
- Lowercasing

In [5]:
def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", str(text))
    text = re.sub(r"\s+", " ", text)
    return text.strip().lower()

train_df["clean_text"] = train_df["raw_text"].apply(clean_text)
test_df["clean_text"]  = test_df["raw_text"].apply(clean_text)

print(train_df[["raw_text", "clean_text", "label"]].head(3))

                                            raw_text  \
0  My question is: Alani earned $45 for 3 hours o...   
1  Q: Yes / no, is the following a verifiable fac...   
2  Read a code snippet and detect if there are an...   

                                          clean_text  label  
0  my question is: alani earned $45 for 3 hours o...      0  
1  q: yes / no, is the following a verifiable fac...      0  
2  read a code snippet and detect if there are an...      0  


### Introducing Convenience splits

In [6]:
from sklearn.model_selection import train_test_split

train_benign_df = train_df[train_df["label"] == 0].copy().reset_index(drop=True)
train_attack_df = train_df[train_df["label"] == 1].copy().reset_index(drop=True)

# Split 65k test into val + test (32k each)
val_df, test_df = train_test_split(
    test_df, test_size=0.5, random_state=42, stratify=test_df["label"]
)
val_df["clean_text"] = val_df["raw_text"].apply(clean_text)

# Rebuilding test convenience splits from the smaller test_df
test_benign_df = test_df[test_df["label"] == 0].copy().reset_index(drop=True)
test_attack_df = test_df[test_df["label"] == 1].copy().reset_index(drop=True)

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")
print(f"\nVal label balance:\n{val_df['label'].value_counts(normalize=True).round(3)}")

Train : 261,738
Val   : 32,708
Test  : 32,708

Val label balance:
label
0    0.506
1    0.494
Name: proportion, dtype: float64


## 4. Binary Classification Baseline

### 4.1 Word-level TF-IDF features

In [7]:
word_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
)

X_train = word_vectorizer.fit_transform(train_df["clean_text"])
X_test  = word_vectorizer.transform(test_df["clean_text"])

y_train = train_df["label"]
y_test  = test_df["label"]

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (261738, 509150)
X_test:  (32708, 509150)


### 4.2 Logistic Regression

In [8]:
lr_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("=== Logistic Regression — Word TF-IDF ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_lr):.4f}")
print()
print(classification_report(y_test, y_pred_lr, digits=4))

=== Logistic Regression — Word TF-IDF ===
Accuracy : 0.9803
F1       : 0.9800

              precision    recall  f1-score   support

           0     0.9758    0.9856    0.9807     16561
           1     0.9850    0.9749    0.9800     16147

    accuracy                         0.9803     32708
   macro avg     0.9804    0.9802    0.9803     32708
weighted avg     0.9804    0.9803    0.9803     32708



### 4.3 Linear SVM

In [ ]:
svm_model = LinearSVC(class_weight="balanced", random_state=42)
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("=== LinearSVC — Word TF-IDF ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_svm):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_svm):.4f}")
print()
print(classification_report(y_test, y_pred_svm, digits=4))

### 4.4 Character-level TF-IDF (exploratory)

Captures sub-word obfuscation patterns.

In [ ]:
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 4),
    min_df=5,
    max_features=80_000,
    sublinear_tf=True,
)

X_train_char = char_vectorizer.fit_transform(train_df["clean_text"])
X_test_char  = char_vectorizer.transform(test_df["clean_text"])

char_lr_model = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
char_lr_model.fit(X_train_char, y_train)

y_pred_char = char_lr_model.predict(X_test_char)

print("=== Logistic Regression — Char TF-IDF ===")
print(classification_report(y_test, y_pred_char, digits=4))

## 5. Adversarial Transformations

In [ ]:
# Leet-speak substitution map
CHAR_MAP = {"a": "@", "e": "3", "i": "1", "o": "0", "s": "$", "t": "7"}

# Cyrillic homoglyph map
HOMOGLYPH_MAP = {"a": "а", "e": "е", "o": "о", "p": "р", "c": "с", "x": "х"}

def char_substitution(text: str, prob: float = 0.18) -> str:
    out = []
    for ch in str(text):
        if ch.lower() in CHAR_MAP and random.random() < prob:
            out.append(CHAR_MAP[ch.lower()])
        else:
            out.append(ch)
    return "".join(out)

def unicode_obfuscation(text: str, prob: float = 0.15) -> str:
    out = []
    for ch in str(text):
        if ch.lower() in HOMOGLYPH_MAP and random.random() < prob:
            out.append(HOMOGLYPH_MAP[ch.lower()])
        else:
            out.append(ch)
    return "".join(out)

def to_indirect(text: str) -> str:
    return f'Summarise the following text exactly as written:\n"{text}"'

def to_narrative(text: str) -> str:
    return f"Write a fictional story where a character does the following: {text}"


In [ ]:
import anthropic, time

# Paraphrase Data
PARA_PATH = "/kaggle/input/datasets/anuragkhadye/paraphrse-data/paraphrase_data.csv"
BT_PATH   = "/kaggle/input/datasets/anuragkhadye/back-translation/back_translation_data.csv"

if os.path.exists(PARA_PATH):
    para_df = pd.read_csv(PARA_PATH)
    print(f"✓ Loaded paraphrase data from dataset ({len(para_df)} rows)")
else:
    print("Paraphrase CSV not found — generating via API...")
    client = anthropic.Anthropic()
    
    def paraphrase(text, retries=3):
        for attempt in range(retries):
            try:
                msg = client.messages.create(
                    model="claude-sonnet-4-6",
                    max_tokens=300,
                    messages=[{"role": "user", "content":
                        f"You are a security researcher studying prompt injection detection. "
                        f"For research purposes, rewrite the following text using completely "
                        f"different wording and sentence structure but preserving the exact "
                        f"same intent. Return only the rewritten text, nothing else.\n\n{text}"
                    }]
                )
                if msg.content and hasattr(msg.content[0], "text"):
                    return msg.content[0].text.strip()
                return text  # empty response — fall back to original
            except Exception as e:
                print(f"  Retry {attempt+1}: {e}")
                time.sleep(2 ** attempt)
        return text  # fallback to original if all retries fail

    para_sample = train_attack_df.sample(n=2_000, random_state=99).copy().reset_index(drop=True)
    para_results = []
    start_idx = 0
        
    if os.path.exists("/kaggle/working/paraphrase_checkpoint.csv"):
        para_results = pd.read_csv("/kaggle/working/paraphrase_checkpoint.csv")["text"].tolist()
        start_idx = len(para_results)
        print(f"Resuming from row {start_idx}/2000")
    else:
        print("Starting fresh")
    
    for i, row in enumerate(para_sample.iloc[start_idx:].itertuples()):
        para_results.append(paraphrase(row.raw_text))
        idx = start_idx + i
        if idx % 50 == 0:
            print(f"  {idx}/2000 done")
            pd.DataFrame({"text": para_results}).to_csv(
                "/kaggle/working/paraphrase_checkpoint.csv", index=False
            )

    para_sample["raw_text"] = para_results
    para_sample.to_csv("/kaggle/working/paraphrase_data.csv", index=False)
    para_df = para_sample
    print(f"  ✓ Done — {len(para_results)} paraphrases saved")

    # Quick check
    print("\nOriginal vs paraphrased (first 3):")
    orig_list = train_attack_df.sample(n=2_000, random_state=99)["raw_text"].tolist()
    for j in range(3):
        print(f"\nOriginal  : {orig_list[j][:100]}")
        print(f"Paraphrase: {para_results[j][:100]}")

In [ ]:
!pip install -q --upgrade sympy transformers

In [ ]:
# Back translation with checkpoint
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

if os.path.exists(BT_PATH):
    bt_df = pd.read_csv(BT_PATH)
    print(f"✓ Loaded back-translation data from dataset ({len(bt_df)} rows)")
else:
    print("Back-translation CSV not found — generating locally...")
    en_fr_tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
    en_fr_mod = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-fr")
    fr_en_tok = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-fr-en")
    fr_en_mod = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-fr-en")

    def back_translate(text):
        try:
            inputs = en_fr_tok(text[:400], return_tensors="pt", truncation=True, max_length=512)
            fr_ids = en_fr_mod.generate(**inputs, max_length=512)
            fr_text = en_fr_tok.decode(fr_ids[0], skip_special_tokens=True)
            inputs = fr_en_tok(fr_text, return_tensors="pt", truncation=True, max_length=512)
            en_ids = fr_en_mod.generate(**inputs, max_length=512)
            return fr_en_tok.decode(en_ids[0], skip_special_tokens=True)
        except Exception:
            return text

    bt_sample = train_attack_df.sample(n=2_000, random_state=77).copy().reset_index(drop=True)
    bt_results = []
    start_idx = 0

    if os.path.exists("/kaggle/working/bt_checkpoint.csv"):
        bt_results = pd.read_csv("/kaggle/working/bt_checkpoint.csv")["text"].tolist()
        start_idx = len(bt_results)
        print(f"  Resuming from row {start_idx}/2000")

    for i, row in enumerate(bt_sample.iloc[start_idx:].itertuples()):
        bt_results.append(back_translate(row.raw_text))
        idx = start_idx + i
        if idx % 100 == 0:
            print(f"  {idx}/2000 done")
            pd.DataFrame({"text": bt_results}).to_csv(
                "/kaggle/working/bt_checkpoint.csv", index=False
            )

    bt_sample["raw_text"] = bt_results
    bt_sample.to_csv("/kaggle/working/back_translation_data.csv", index=False)
    bt_df = bt_sample
    print(f"  ✓ Done — {len(bt_results)} back-translations saved")

print(f"\npara_df: {len(para_df)} rows | bt_df: {len(bt_df)} rows")

In [ ]:
# Final Data check
sample = train_attack_df["raw_text"].iloc[3]
print("ORIGINAL                   :", sample[:80])
print("CHAR SUB                   :", char_substitution(sample)[:80])
print("UNICODE                    :", unicode_obfuscation(sample)[:80])
print("INDIRECT                   :", to_indirect(sample)[:80])
print("NARRATIVE                  :", to_narrative(sample)[:80])
print("PARAPHRASE(normal)         :", para_df["raw_text"].iloc[0][:80])
print("PARAPHRASE(parphrased)     :", para_df["clean_text"].iloc[0][:80])
print("BACK-TRANS(normal)         :", bt_df["raw_text"].iloc[0][:80])
print("BACK-TRANS(back_translated):", bt_df["clean_text"].iloc[0][:80])

In [ ]:
ADV_N = 10000
benign_eval   = test_benign_df.sample(n=ADV_N, random_state=42).copy()
attack_sample = test_attack_df.sample(n=ADV_N, random_state=42).copy()

# Build one adversarial variant per attack type
def make_adv_variant(base_df, transform_fn, attack_label):
    df = base_df.copy()
    df["raw_text"]    = df["raw_text"].apply(transform_fn)
    df["attack_type"] = attack_label
    return df

char_adv      = make_adv_variant(attack_sample, char_substitution, "char_substitution")
unicode_adv   = make_adv_variant(attack_sample, unicode_obfuscation, "unicode_obfuscation")
indirect_adv  = make_adv_variant(attack_sample, to_indirect, "indirect_injection")
narrative_adv = make_adv_variant(attack_sample, to_narrative, "narrative_roleplay")

# Build mixed sets (10k benign + 10k adversarial each)
def build_mixed_set(benign_df, adv_df):
    combined = pd.concat([benign_df, adv_df], ignore_index=True)
    combined["clean_text"] = combined["raw_text"].apply(clean_text)
    return combined.sample(frac=1, random_state=42).reset_index(drop=True)

char_mixed      = build_mixed_set(benign_eval, char_adv)
unicode_mixed   = build_mixed_set(benign_eval, unicode_adv)
indirect_mixed  = build_mixed_set(benign_eval, indirect_adv)
narrative_mixed = build_mixed_set(benign_eval, narrative_adv)

# Evaluate binary models on each mixed set
def evaluate_mixed(df, model, vec, name):
    X     = vec.transform(df["clean_text"])
    y     = df["label"]
    y_hat = model.predict(X)
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    print(classification_report(y, y_hat, digits=4))
    return f1_score(y, y_hat, average="macro")

results = []
for model_name, model in [("LR", lr_model), ("SVM", svm_model)]:
    for attack_name, df_mix in [
        ("char_substitution",   char_mixed),
        ("unicode_obfuscation", unicode_mixed),
        ("indirect_injection",  indirect_mixed),
        ("narrative_roleplay",  narrative_mixed),
    ]:
        macro_f1 = evaluate_mixed(df_mix, model, word_vectorizer,
                                  f"{model_name} — {attack_name}")
        results.append({"Model": model_name, "Attack": attack_name, "Macro_F1": round(macro_f1, 4)})

results_df = pd.DataFrame(results)
print("\n=== Summary ===")
print(results_df.to_string(index=False))

In [ ]:
AUG_N = 30_000

train_attack_sample = train_attack_df.sample(n=AUG_N, random_state=42).copy()

char_aug = train_attack_sample.copy()
char_aug["raw_text"] = char_aug["raw_text"].apply(char_substitution)

unicode_aug = train_attack_sample.copy()
unicode_aug["raw_text"] = unicode_aug["raw_text"].apply(unicode_obfuscation)

indirect_aug = train_attack_sample.copy()
indirect_aug["raw_text"] = indirect_aug["raw_text"].apply(to_indirect)

aug_train_df = pd.concat(
    [train_df, char_aug, unicode_aug, indirect_aug],
    ignore_index=True,
).sample(frac=1, random_state=42).reset_index(drop=True)

aug_train_df["clean_text"] = aug_train_df["raw_text"].apply(clean_text)

print(f"Original train : {len(train_df):,}")
print(f"Augmented train: {len(aug_train_df):,}")
print(aug_train_df["label"].value_counts())

In [ ]:
aug_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True,
)

X_train_aug = aug_vectorizer.fit_transform(aug_train_df["clean_text"])
X_test_aug  = aug_vectorizer.transform(test_df["clean_text"])

svm_adv_model = LinearSVC(class_weight="balanced", random_state=42)
svm_adv_model.fit(X_train_aug, aug_train_df["label"])

y_pred_adv = svm_adv_model.predict(X_test_aug)
print("=== Adversarially-Trained SVM — Clean Test ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_adv):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_adv):.4f}")
print(classification_report(y_test, y_pred_adv, digits=4))

In [ ]:
## 7.2 Adv-trained SVM on mixed attack sets
print("=== Adversarially-Trained SVM — Mixed Attack Sets ===")
for attack_name, df_mix in [
    ("char_substitution",   char_mixed),
    ("unicode_obfuscation", unicode_mixed),
    ("indirect_injection",  indirect_mixed),
    ("narrative_roleplay",  narrative_mixed),
]:
    evaluate_mixed(df_mix, svm_adv_model, aug_vectorizer,
                   f"Adv-SVM — {attack_name}")

In [ ]:
LABEL_MAP_7 = {
    "benign":               0,
    "direct_jailbreak":     1,
    "indirect_injection":   2,
    "narrative_roleplay":   3,
    "obfuscation":          4,
    "paraphrase_jailbreak": 5,
    "back_translation":     6,
}

N_TRAIN = 1_600
N_TEST  = 400

# Split para_df and bt_df into train/test
para_train, para_test = train_test_split(para_df, test_size=N_TEST, random_state=42)
bt_train, bt_test     = train_test_split(bt_df,   test_size=N_TEST, random_state=42)

# Shuffle attack pools once, slice into 4 non-overlapping windows
train_attack_shuffled = train_attack_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_attack_shuffled  = test_attack_df.sample(frac=1, random_state=42).reset_index(drop=True)

assert len(train_attack_shuffled) >= 4 * N_TRAIN, "Insufficient train attack samples"
assert len(test_attack_shuffled)  >= 4 * N_TEST,  "Insufficient test attack samples"

# 4 slices from the attack pool (each gets different prompts)
train_slices = {
    "direct_jailbreak":   train_attack_shuffled.iloc[0         : N_TRAIN  ].copy(),
    "indirect_injection": train_attack_shuffled.iloc[N_TRAIN   : 2*N_TRAIN].copy(),
    "narrative_roleplay": train_attack_shuffled.iloc[2*N_TRAIN : 3*N_TRAIN].copy(),
    "obfuscation":        train_attack_shuffled.iloc[3*N_TRAIN : 4*N_TRAIN].copy(),
}
test_slices = {
    "direct_jailbreak":   test_attack_shuffled.iloc[0        : N_TEST  ].copy(),
    "indirect_injection": test_attack_shuffled.iloc[N_TEST   : 2*N_TEST].copy(),
    "narrative_roleplay": test_attack_shuffled.iloc[2*N_TEST : 3*N_TEST].copy(),
    "obfuscation":        test_attack_shuffled.iloc[3*N_TEST : 4*N_TEST].copy(),
}

# Overlap checks
for name, slices in [("train", train_slices), ("test", test_slices)]:
    idx_sets = [set(s.index) for s in slices.values()]
    assert all(
        a.isdisjoint(b)
        for i, a in enumerate(idx_sets)
        for b in idx_sets[i+1:]
    ), f"Overlap detected in {name} partitions!"
print("✓ All train partitions are non-overlapping")
print("✓ All test partitions are non-overlapping")

# Apply transforms to the 4 pool-based classes
TRANSFORM = {
    "direct_jailbreak":   lambda t: t,
    "indirect_injection": to_indirect,
    "narrative_roleplay": to_narrative,
    "obfuscation":        lambda t: unicode_obfuscation(char_substitution(t)),
}

for cls_name in TRANSFORM:
    train_slices[cls_name]["raw_text"] = train_slices[cls_name]["raw_text"].apply(TRANSFORM[cls_name])
    test_slices[cls_name]["raw_text"]  = test_slices[cls_name]["raw_text"].apply(TRANSFORM[cls_name])

# Benign samples
mc_train_benign = train_benign_df.sample(n=N_TRAIN, random_state=42).copy()
mc_test_benign  = test_benign_df.sample(n=N_TEST,   random_state=42).copy()

# Tag everything with multiclass labels
def tag(df, cls_name):
    df = df[["raw_text"]].copy()
    df["mc_label"] = LABEL_MAP_7[cls_name]
    return df

train_parts = [tag(mc_train_benign, "benign")]
test_parts  = [tag(mc_test_benign, "benign")]

for cls_name in TRANSFORM:
    train_parts.append(tag(train_slices[cls_name], cls_name))
    test_parts.append(tag(test_slices[cls_name], cls_name))

# Add paraphrase and back-translation classes
train_parts.append(tag(para_train, "paraphrase_jailbreak"))
test_parts.append(tag(para_test, "paraphrase_jailbreak"))
train_parts.append(tag(bt_train, "back_translation"))
test_parts.append(tag(bt_test, "back_translation"))

# Build final DataFrames
train_mc_df = pd.concat(train_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
test_mc_df  = pd.concat(test_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

train_mc_df["clean_text"] = train_mc_df["raw_text"].apply(clean_text)
test_mc_df["clean_text"]  = test_mc_df["raw_text"].apply(clean_text)

print("Train class distribution:")
print(train_mc_df["mc_label"].value_counts().sort_index())
print(f"\nTest class distribution:")
print(test_mc_df["mc_label"].value_counts().sort_index())
print(f"\nTotal — Train: {len(train_mc_df):,} | Test: {len(test_mc_df):,}")

In [ ]:
## 8.1 Vectorise and evaluate multiclass models
mc_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2), min_df=3, max_df=0.95, sublinear_tf=True,
)

X_train_mc = mc_vectorizer.fit_transform(train_mc_df["clean_text"])
X_test_mc  = mc_vectorizer.transform(test_mc_df["clean_text"])
y_train_mc = train_mc_df["mc_label"]
y_test_mc  = test_mc_df["mc_label"]

CLASS_NAMES_7 = list(LABEL_MAP_7.keys())

def eval_mc(model, X, y, name):
    y_hat = model.predict(X)
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"{'='*60}")
    print(f"Accuracy : {accuracy_score(y, y_hat):.4f}")
    print(f"Macro F1 : {f1_score(y, y_hat, average='macro'):.4f}")
    print()
    print(classification_report(y, y_hat, target_names=CLASS_NAMES_7, digits=4))
    print("Confusion Matrix:")
    print(confusion_matrix(y, y_hat))

lr_mc = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr_mc.fit(X_train_mc, y_train_mc)
eval_mc(lr_mc, X_test_mc, y_test_mc, "Logistic Regression — 7-class")

svm_mc = LinearSVC(class_weight="balanced", random_state=42)
svm_mc.fit(X_train_mc, y_train_mc)
eval_mc(svm_mc, X_test_mc, y_test_mc, "LinearSVC — 7-class")

In [ ]:
from huggingface_hub import login
login(token=secrets.get_secret("HF_TOKEN"))

In [ ]:
## 9. BERT Fine-Tuning — Binary Classification
import torch
import numpy as np
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

BERT_SAVE_PATH = "/kaggle/input/models/anuragkhadye/bert/pytorch/default/1/bert_binary_final"
MODEL_NAME = "distilbert-base-uncased"

if os.path.exists(BERT_SAVE_PATH):
    print("✓ Loading saved BERT model — skipping training")
    model = AutoModelForSequenceClassification.from_pretrained(BERT_SAVE_PATH).to(device)
    tokenizer = AutoTokenizer.from_pretrained(BERT_SAVE_PATH)
else:
    print("BERT model not found — training from scratch...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    def tokenise(df):
        return tokenizer(
            df["clean_text"].tolist(),
            padding=True, truncation=True, max_length=128, return_tensors="pt"
        )

    print("Tokenising train...")
    train_enc = tokenise(train_df)
    print("Tokenising val...")
    val_enc   = tokenise(val_df)

    class PromptDataset(torch.utils.data.Dataset):
        def __init__(self, encodings, labels):
            self.encodings = encodings
            self.labels = labels
        def __len__(self):
            return len(self.labels)
        def __getitem__(self, idx):
            item = {k: v[idx] for k, v in self.encodings.items()}
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
            return item

    train_dataset = PromptDataset(train_enc, train_df["label"].tolist())
    val_dataset   = PromptDataset(val_enc,   val_df["label"].tolist())

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=2
    ).to(device)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)
        return {
            "accuracy": accuracy_score(labels, preds),
            "f1": f1_score(labels, preds),
        }

    training_args = TrainingArguments(
        output_dir="/kaggle/working/bert-binary",
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=64,
        num_train_epochs=3,
        weight_decay=0.01,
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        fp16=True,
        logging_steps=500,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    trainer.train()
    trainer.save_model("/kaggle/working/bert-binary-final")
    tokenizer.save_pretrained("/kaggle/working/bert-binary-final")
    print("✓ Model trained and saved")


In [ ]:
import os
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        if f == "config.json":
            print(os.path.join(root, f))

In [ ]:
# Binary BERT

class PromptDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

trainer = Trainer(model=model)

# Clean test
print("Evaluating clean test...")
test_enc = tokenizer(test_df["clean_text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")
test_dataset = PromptDataset(test_enc, test_df["label"].tolist())

predictions = trainer.predict(test_dataset)
y_pred_bert = np.argmax(predictions.predictions, axis=-1)

print("=== BERT — Clean Test ===")
print(f"Accuracy : {accuracy_score(y_test, y_pred_bert):.4f}")
print(f"F1       : {f1_score(y_test, y_pred_bert):.4f}")
print(classification_report(y_test, y_pred_bert, digits=4))

In [ ]:
# Adversarial evaluation
def evaluate_bert_mixed(df, name):
    enc = tokenizer(df["clean_text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")
    ds = PromptDataset(enc, df["label"].tolist())
    preds = trainer.predict(ds)
    y_hat = np.argmax(preds.predictions, axis=-1)
    y = df["label"].values
    macro = f1_score(y, y_hat, average="macro")
    print(f"\n{'='*55}")
    print(f"  BERT — {name}")
    print(f"{'='*55}")
    print(classification_report(y, y_hat, digits=4))
    return macro

bert_results = []
for attack_name, df_mix in [
    ("char_substitution",   char_mixed),
    ("unicode_obfuscation", unicode_mixed),
    ("indirect_injection",  indirect_mixed),
    ("narrative_roleplay",  narrative_mixed),
]:
    f = evaluate_bert_mixed(df_mix, attack_name)
    bert_results.append({"Attack": attack_name, "BERT_F1": round(f, 4)})

bert_results_df = pd.DataFrame(bert_results)
print("\n=== BERT Adversarial Summary ===")
print(bert_results_df.to_string(index=False))


In [ ]:
## 10. BERT Multiclass (7-class)

# Tokenise multiclass splits
print("Tokenising multiclass train...")
mc_train_enc = tokenizer(train_mc_df["clean_text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")
print("Tokenising multiclass test...")
mc_test_enc = tokenizer(test_mc_df["clean_text"].tolist(), padding=True, truncation=True, max_length=128, return_tensors="pt")

mc_train_dataset = PromptDataset(mc_train_enc, train_mc_df["mc_label"].tolist())
mc_test_dataset  = PromptDataset(mc_test_enc,  test_mc_df["mc_label"].tolist())

# Load fresh model with 7 labels
mc_model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=7
).to(device)

def compute_metrics_mc(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
    }

mc_training_args = TrainingArguments(
    output_dir="/kaggle/working/bert-multiclass",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=True,
    logging_steps=100,
    report_to="none",
)

mc_trainer = Trainer(
    model=mc_model,
    args=mc_training_args,
    train_dataset=mc_train_dataset,
    eval_dataset=mc_test_dataset,
    compute_metrics=compute_metrics_mc,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

mc_trainer.train()

# Evaluate
mc_preds = mc_trainer.predict(mc_test_dataset)
y_hat_mc = np.argmax(mc_preds.predictions, axis=-1)
y_true_mc = test_mc_df["mc_label"].values

CLASS_NAMES_7 = list(LABEL_MAP_7.keys())

print(f"\n{'='*60}")
print(f"  BERT — 7-class Multiclass")
print(f"{'='*60}")
print(f"Accuracy : {accuracy_score(y_true_mc, y_hat_mc):.4f}")
print(f"Macro F1 : {f1_score(y_true_mc, y_hat_mc, average='macro'):.4f}")
print()
print(classification_report(y_true_mc, y_hat_mc, target_names=CLASS_NAMES_7, digits=4))
print("Confusion Matrix:")
print(confusion_matrix(y_true_mc, y_hat_mc))

# Save
mc_trainer.save_model("/kaggle/working/bert-multiclass-final")
tokenizer.save_pretrained("/kaggle/working/bert-multiclass-final")
print("\n✓ Multiclass model saved")